# ☁️ Start here: take the course on Google Colab

**Run this notebook once per Colab account** (and again whenever you want the latest course). It takes 3–5 minutes.

> ### 📌 You do NOT upload anything
> This notebook copies the course into **your** Google Drive by itself: when it finishes, the course is in **My Drive/langchain-langgraph-mastery**. Then you open each lesson **from Drive** ([What's next](#whats-next) at the bottom shows how). Running *this* notebook from GitHub is fine.

| | |
|---|---|
| ☁️ Google Colab | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diego28e/langchain-langgraph-mastery/blob/main/colab/START_HERE.ipynb) |
| ⏱️ Time | 3–5 min (first run), under 1 min after that |
| 🔑 Keys | Optional now: add them later in **Colab Secrets** (🔑 in the left sidebar), Step 4 |
| 🧠 Runtime | The default **Python 3 / CPU** runtime is enough (a GPU is only for Lesson 14.04's Colab notebook) |
| 📄 Guide | [docs/COLAB.md](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md): what runs on Colab, what needs your PC, troubleshooting |

**What it does:**
1. Mounts your **Google Drive** and clones the course into `MyDrive/langchain-langgraph-mastery` (or updates it with `git pull`). Your clone and everything the lessons write (`data/outputs/`) stay in Drive between sessions.
2. Installs the course into this runtime with versions tested on Colab (`pip install -e ".[colab]"` plus `constraints/colab-2026-10.txt`).
3. Loads your **Colab Secrets** (API keys) and shows a ✅/❌ table.

**How:** **Runtime → Run all**. When Google asks about Drive, tick **Select all** (Step 2 explains). If Step 3 stops with **"Restart once"**, do **Runtime → Restart session**, then **Runtime → Run all** again: the second run skips the install and finishes. That's all.

> **Prefer your own PC?** Close this notebook and follow [00-02-create-course-env.md](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/modules/00-foundations/00-02-create-course-env.md). You can mix both: most lessons run on either, a few need your PC ([the list](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md#per-module)).

## Step 1 · Settings

Leave these as they are unless you forked the course to your own GitHub account.

In [ ]:
# Step 1: Settings (edit only for your own fork or another branch).
REPO_URL = "https://github.com/diego28e/langchain-langgraph-mastery.git"
BRANCH = "main"
USE_DRIVE = True  # True: keep the clone in Google Drive (survives sessions). False: /content (deleted when the runtime ends).
UPDATE_MY_EDITED_LESSONS = False  # True: replace lessons you edited with the course's newer version (yours is backed up).

## Step 2 · Mount Drive and get the course

The first time, this **clones** the course. Later runs **update** it, and **keep your work**: open lessons from Drive and work right in them (add cells, write code); your work saves automatically and updates keep it.
- Lessons you **edited** stay exactly as you left them, in place. If the course changed one of them, this cell tells you.
- Lessons you only **ran** (Colab saved their outputs) are simply updated.

> **⚠️ When Google asks for Drive access, allow all of it.** Colab first asks *"Permit this notebook to access your Google Drive files?"*: click **Connect to Google Drive**. A Google window opens (it may say **Google Drive for desktop**: that is what Colab uses to mount Drive):
> 1. Choose **the same Google account** you use for Colab.
> 2. On the permissions screen, tick **Select all**. Ticking only some boxes makes the mount fail with *"credential propagation was unsuccessful"*.
> 3. Click **Continue**.
>
> No window? Your browser blocked the pop-up: allow pop-ups for `colab.research.google.com` and run this cell again. If the mount still fails, this cell prints the fix ([troubleshooting](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md#update)).

In [ ]:
# Step 2: Mount Google Drive, then clone the course (first time) or update it (later runs).
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO_NAME = "langchain-langgraph-mastery"
BASE = Path("/content/drive/MyDrive") if USE_DRIVE else Path("/content")
REPO_ROOT = BASE / REPO_NAME
EDITS = BASE / f"{REPO_NAME}-my-edits"  # Safety copies of the files you edited, made before each update.


def git(*args: str, repo: Path | None = None) -> tuple[int, str]:
    """Run git (in `repo` via git -C, so a missing folder is a git error, not a traceback): (exit code, output)."""
    command = ["git", *(["-C", str(repo)] if repo else []), *args]
    try:
        result = subprocess.run(command, capture_output=True, text=True)
    except OSError as error:
        return 1, str(error)
    return result.returncode, (result.stdout + result.stderr).strip()


def git_text(*args: str) -> str | None:
    """The stdout of git in REPO_ROOT (a file at a commit, a list of commits), or None if git failed."""
    try:
        result = subprocess.run(["git", "-C", str(REPO_ROOT), *args], capture_output=True)
    except OSError:
        return None
    return result.stdout.decode("utf-8", "replace") if result.returncode == 0 else None


def sources(text: str | None) -> list[tuple[str, str]] | None:
    """A notebook's cells as (type, code or text): what you write. Outputs, counts and metadata are left out."""
    try:
        found = []
        for cell in json.loads(text or "")["cells"]:
            source = cell.get("source", "")
            source = "".join(source) if isinstance(source, list) else str(source)
            found.append((cell.get("cell_type", ""), source.replace("\r\n", "\n").rstrip()))
        return found
    except (ValueError, KeyError, TypeError, AttributeError):
        return None


def only_colab_saves(path: str) -> bool:
    """True when this notebook's code and text equal a committed version (HEAD or one of its last 30 versions):
    Colab saved it with outputs or settings, or saved an older course copy back over an update. Not your edits."""
    if not path.endswith(".ipynb"):
        return False
    try:
        mine = sources((REPO_ROOT / path).read_text(encoding="utf-8"))
    except (OSError, UnicodeDecodeError):
        return False
    if mine is None:
        return False
    revisions = ["HEAD", *(git_text("log", "-n", "30", "--format=%H", "--", path) or "").split()]
    return any(sources(git_text("show", f"{revision}:{path}")) == mine for revision in revisions)


def update_course() -> None:
    """Update the clone and keep your work: files you edited stay in place, Colab-only saves are updated.

    Each update: fetch; if there is something new, copy your edited files to EDITS (a safety net, checked byte for
    byte), reset everything that changed, fast-forward, then write your edited versions back where they were.
    """
    git("config", "core.fileMode", "false", repo=REPO_ROOT)  # Drive drops file modes: not changes.
    code, out = git("fetch", "origin", BRANCH, repo=REPO_ROOT)
    old, new = git_text("rev-parse", "HEAD"), git_text("rev-parse", "FETCH_HEAD")
    if code != 0 or not new:
        print("⚠️ Could not reach GitHub, so the course was not updated (your copy still works). Run this cell later.")
        print("  ", out.replace("\n", "\n   "))
        return
    if old == new:
        print("✅ Course is up to date:", REPO_ROOT)
        return
    if git("merge-base", "--is-ancestor", "HEAD", "FETCH_HEAD", repo=REPO_ROOT)[0] != 0:
        print("❌ Could not update the course: your clone has commits of its own (git commit was run in it), so git")
        print("   can't simply move it forward. Nothing was changed: your copy and your work still work as they are.")
        print(f"   To get the update: in https://drive.google.com/drive/my-drive rename '{REPO_NAME}' to '{REPO_NAME}-old'")
        print("   (that keeps everything in it), then Runtime → Run all here for a fresh copy.")
        return
    changed = [path for path in (git_text("diff", "--name-only", "-z", "HEAD") or "").split("\0") if path]
    saves = [path for path in changed if not (REPO_ROOT / path).exists() or only_colab_saves(path)]
    edits = [path for path in changed if path not in saves]
    upstream = set((git_text("diff", "--name-only", "-z", "HEAD", "FETCH_HEAD", "--", *edits) or "").split("\0"))
    newer = [path for path in edits if path in upstream]
    kept = {path: (REPO_ROOT / path).read_bytes() for path in edits}
    backup = EDITS / time.strftime("%Y-%m-%d_%H-%M-%S_UTC", time.gmtime())
    try:
        for path, data in kept.items():
            (backup / path).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(REPO_ROOT / path, backup / path)
            if (backup / path).read_bytes() != data:
                raise OSError(f"the safety copy of {path} is incomplete")
    except OSError as error:
        print("⚠️ Could not make a safety copy of your work in", backup, "·", error)
        print("   So nothing was changed or updated: your copy and your work stay exactly as they are.")
        print("   Run this cell again in a minute.")
        return
    code, out = git("checkout", "HEAD", "--", *changed, repo=REPO_ROOT) if changed else (0, "")
    if code == 0:
        code, out = git("merge", "--ff-only", "FETCH_HEAD", repo=REPO_ROOT)
    replace = set(newer) if code == 0 and globals().get("UPDATE_MY_EDITED_LESSONS", False) else set()
    lost = []
    for path, data in kept.items():  # Put your work back where it was, whatever happened above.
        if path in replace:
            continue
        try:
            (REPO_ROOT / path).parent.mkdir(parents=True, exist_ok=True)
            (REPO_ROOT / path).write_bytes(data)
            if (REPO_ROOT / path).read_bytes() != data:
                raise OSError("written back incompletely")
        except OSError:
            lost.append(path)
    if code == 0:
        print("✅ Course updated.")
    else:
        print("❌ Could not update the course. Your copy and your work still work as they are.")
        if "untracked working tree files" in out:
            print("   Files you created have the same names as new course files: rename them, then run this cell again.")
        print("  ", out.replace("\n", "\n   "))
    if kept and len(replace) < len(kept):
        print(f"📝 Kept your work in {len(kept) - len(replace)} lesson(s):", ", ".join(p for p in kept if p not in replace))
    for path in newer:
        if path in replace:
            print(f"🔄 Updated {path} to the newer version. Your version: My Drive/{(backup / path).relative_to(BASE).as_posix()}")
        elif code == 0:
            print(f"ℹ️ A newer version of {path} exists. To get it, rename your copy in Drive (for example add '-mine'),")
            print("   then run START_HERE again. Or set UPDATE_MY_EDITED_LESSONS = True in Step 1 (your version is backed up).")
    for path in lost:
        print(f"⚠️ Could not put back your version of {path}. It is safe in My Drive/{(backup / path).relative_to(BASE).as_posix()}")


def clone_visible(root: Path, seconds: int = 30) -> bool:
    """Drive can take a few seconds to show a folder git just wrote: wait until root/.git appears."""
    for _ in range(seconds):
        if (root / ".git").is_dir():
            return True
        try:
            os.listdir(root.parent)  # Re-read the folder from Drive.
        except OSError:
            pass
        time.sleep(1)
    return (root / ".git").is_dir()


def clone_missing(why: str) -> None:
    """Explain a clone Drive doesn't show (instead of a subprocess traceback), then stop Run all."""
    print(f"❌ {why}, but Drive doesn't show {REPO_ROOT / '.git'} after 30 seconds.")
    print("   Usually Drive is just slow to show a fresh clone. To fix it:")
    print("   1. Wait a minute, then run this cell again: it picks up the clone (or clones it if it isn't there).")
    print(f"   2. Still failing? Open https://drive.google.com/drive/my-drive: if '{REPO_NAME}' is there but has no")
    print(f"      files, rename it (for example to {REPO_NAME}-old) and run this cell again.")
    print("   3. Then Runtime → Disconnect and delete runtime, and Runtime → Run all.")
    raise RuntimeError(f"The course clone is not visible at {REPO_ROOT} yet: follow the steps above.") from None


if not IN_COLAB:
    print("ℹ️ This notebook sets up Google Colab. On your PC, follow modules/00-foundations/00-02-create-course-env.md.")
else:
    if USE_DRIVE and not BASE.is_dir():
        from google.colab import drive

        try:
            drive.mount("/content/drive")  # Google asks you to allow access: tick "Select all".
        except Exception as error:  # Usually google.colab's MessageError: "credential propagation was unsuccessful".
            print("❌ Google Drive did not mount:", error)
            print("   Usual cause: only some of the Drive permissions were ticked. To fix it:")
            print("   1. Runtime → Disconnect and delete runtime.")
            print("   2. Runtime → Run all. In the Google window, choose your Colab account, tick Select all, then Continue.")
            print("      No window? Allow pop-ups for colab.research.google.com.")
            print("   3. Still failing? Open https://myaccount.google.com/connections, choose 'Google Drive for desktop',")
            print("      remove its access (Google then asks for every permission again), and repeat steps 1 and 2.")
            print("   Last resort: USE_DRIVE = False in Step 1 clones into /content instead. It works in this runtime only,")
            print("   is deleted when the runtime resets, and lessons (each opens its own runtime) won't find it.")
            raise RuntimeError("Drive did not mount: follow the steps above, then run all again.") from None
    if (REPO_ROOT / ".git").is_dir() or (REPO_ROOT.exists() and clone_visible(REPO_ROOT, 15)):
        update_course()
    elif REPO_ROOT.exists():
        raise FileExistsError(f"{REPO_ROOT} exists but has no .git folder. Just cloned it? Drive may still be showing"
                              " it: wait a minute and run this cell again. Otherwise rename it in Drive and run again.")
    else:
        print("⏳ Cloning the course into", REPO_ROOT, "...")
        code, out = git("clone", "--branch", BRANCH, REPO_URL, str(REPO_ROOT))
        if code != 0:
            raise RuntimeError("git clone failed:\n" + out)
        if not clone_visible(REPO_ROOT):
            clone_missing("git cloned the course")
        print("✅ Course cloned:", REPO_ROOT)
    if not clone_visible(REPO_ROOT):
        clone_missing("git finished")
    code, out = git("config", "core.fileMode", "false", repo=REPO_ROOT)  # Drive drops file modes: not changes.
    if code != 0:
        print("⚠️ Could not set core.fileMode (harmless: git may list every file as changed):", out)

## Step 3 · Install the course into this runtime

Each Colab notebook runs in its **own runtime**, so every lesson's setup cell repeats this step automatically the first time you run it (about 1–3 minutes; after that it is skipped until the runtime is recycled). Here you do it once to check everything works.

If this cell stops with **"Restart once"**: **Runtime → Restart session**, then **Runtime → Run all**.

In [ ]:
# Step 3: Install the course (once per runtime) with the Colab-tested versions, then load Colab Secrets.
if IN_COLAB:
    import importlib

    _src = str((REPO_ROOT / "src").resolve())
    sys.path[:] = [_src, *(p for p in sys.path if p != _src)]  # This clone's code first.
    for _name in [n for n in sys.modules if n == "mastery" or n.startswith("mastery.")]:
        del sys.modules[_name]  # Step 2 may have just updated the course: never reuse an older import.
    importlib.invalidate_caches()
    from mastery.colab import prepare_runtime

    prepare_runtime(REPO_ROOT, secrets=())  # Step 4 reads the keys. Raises RestartRequired the first time if Colab had already imported an upgraded package.

## Step 4 · Your API keys: Colab Secrets

Colab keeps keys out of your notebooks: click **🔑 Secrets** in the left sidebar → **Add new secret**, use the **same names as in `.env.example`**, and switch **Notebook access** on. The course reads them by itself (`load_env()` copies them into the environment and never overwrites a variable you set yourself).

| Secret | Needed for |
|---|---|
| `GROQ_API_KEY` | Most 💸 cells (free tier, Lesson 00.04 shows how to get it) |
| `LANGSMITH_API_KEY` | Traces (Lesson 01.08), optional |
| `TAVILY_API_KEY` | Web search lessons, optional |
| `GOOGLE_API_KEY` | Gemini lessons, optional |

Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs. After you add a secret, run the next cell again (no restart needed). [All the names](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md#secrets)

In [ ]:
# Step 4: Which keys does this runtime see? (Names only, never values.)
if IN_COLAB:
    from mastery.config import is_set, load_colab_secrets

    KEYS = ("GROQ_API_KEY", "LANGSMITH_API_KEY", "TAVILY_API_KEY", "GOOGLE_API_KEY")
    loaded = load_colab_secrets(KEYS, force=True)  # force: re-read after you add a secret
    print("Loaded from Colab Secrets:", ", ".join(loaded) or "(none yet)")
    for name in KEYS:
        print(f"  {'✅' if is_set(name) else '➖'} {name}")

## Step 5 · Verify

The same idea as Lesson 00.02: which Python and which packages this runtime really uses.

In [ ]:
# Step 5: A verification table for this runtime.
import platform
from importlib import metadata

from IPython.display import Markdown, display


def installed(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "❌ not installed"


if IN_COLAB:
    import mastery

    in_repo = Path(mastery.__file__).resolve().is_relative_to(REPO_ROOT.resolve())
    rows = [
        ("Python", platform.python_version(), "✅" if sys.version_info >= (3, 11) else "❌ needs 3.11+"),
        ("Course clone", str(REPO_ROOT), "✅ in Drive" if USE_DRIVE else "⚠️ temporary (/content)"),
        ("mastery", f"{mastery.__version__} from {Path(mastery.__file__).parent}", "✅" if in_repo else "❌ restart the session"),
    ]
    for package in ("langchain", "langchain-core", "langgraph", "langchain-groq", "langsmith", "pandas", "pytest"):
        version = installed(package)
        rows.append((package, version, "❌" if version.startswith("❌") else "✅"))
    rows.append(("GROQ_API_KEY", "set" if is_set("GROQ_API_KEY") else "not set", "✅" if is_set("GROQ_API_KEY") else "➖ optional"))
    table = "| Check | Value | |\n|---|---|---|\n" + "\n".join(f"| {a} | `{b}` | {c} |" for a, b, c in rows)
    display(Markdown(table))
    if USE_DRIVE:
        from mastery.colab import drive_file_id, drive_folder_url

        print(f"✅ Done. You do NOT upload anything: the course is now in My Drive/{REPO_NAME}.")
        print("➡️ Open lessons from Drive:", drive_folder_url(REPO_ROOT / "modules"))
        for lesson in ("00-02-verify-environment.ipynb", "00-03-jupyter-fundamentals.ipynb"):  # Or jump straight in.
            file_id = drive_file_id(REPO_ROOT / "modules" / "00-foundations" / lesson)
            if file_id:
                print(f"   {lesson}: https://colab.research.google.com/drive/{file_id}")
        print(f"   drive.google.com → My Drive → {REPO_NAME} → modules → a module → right-click the .ipynb")
        print("   → Open with → Google Colaboratory. The title bar shows the Drive icon, not 'Save in GitHub'.")
        print("   Next time: File → Open notebook → Recent. (Colab's 'Google Drive' tab only lists notebooks")
        print("   you opened recently, so don't use it to browse.) From Drive, your work saves and Colab")
        print("   remembers the Secrets access you grant each notebook.")
    else:
        print("⚠️ USE_DRIVE = False: the clone is temporary and lessons (each in its own runtime) won't find it.")

**You should now see** ✅ on every row (➖ for keys you haven't added yet). A ❌ on `mastery` means this kernel had already imported another copy: **Runtime → Restart session**, then **Run all**.

---
<a id="whats-next"></a>
## ➡️ What's next: open the lessons from Drive

**You do NOT upload anything.** The course is now in **My Drive/langchain-langgraph-mastery** (Step 2 put it there; Step 5 printed a link to its `modules` folder).

1. **Open a lesson from Drive:** [drive.google.com](https://drive.google.com/drive/my-drive) → **My Drive** → **langchain-langgraph-mastery** → **modules** → a module folder → right-click the lesson's `.ipynb` → **Open with** → **Google Colaboratory**.
2. **Check the title bar:** it shows the **Drive icon**. If you see **"Save in GitHub"**, you opened the GitHub copy: your edits won't be saved, so close it and open the lesson from Drive.
3. **Next time:** **File → Open notebook → Recent** lists the lessons you've opened.
4. **Run it top to bottom.** Its first code cell finds your clone, installs the course into that runtime (once) and reads only the keys that lesson uses. If it asks you to **restart once**, do it and run all again.
5. **Work right in it.** Add cells, write code: your work saves automatically, and running this notebook again to update keeps it.

> **Don't browse with Colab's File → Open notebook → Google Drive tab:** it only lists notebooks you opened recently (at first just "Untitled"), not your folders.
>
> **Why from Drive?** Your work saves into your clone, and Colab remembers the Secrets access you grant each notebook. The **Open in Colab** badges open the read-only GitHub copy: fine for a look, not for working.

Lessons marked **❌ Not available on Colab** (Docker, Ollama, Studio) need your PC or the devcontainer: [the list and the setup](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md#per-module).

Good first lessons, in `modules/00-foundations/`: `00-03-jupyter-fundamentals.ipynb` · `00-04-secrets-and-env.ipynb`, then `modules/01-models-messages-prompts/01-01-chat-models.ipynb`.